# PapyrusLab E03 — run `infer-46527-s43-zp2` (segmento `pherc0814-46527`, offset `zp2`)

Notebook generato da `scripts/build_e03_notebooks.py` (non modificare a mano). Piano congelato: `docs/plans/2026-09-07-e03-tolleranza-offset-z.md`.
Ogni controllo di arresto del piano è un'asserzione: se fallisce, il run si ferma e il log dice dove.
Fanno fede il piano e `configs/e03/offsets.json`; questo generatore ne è solo l'espansione meccanica.

- Output persistiti: `/kaggle/working/e03/out` e `/kaggle/working/e03/logs`
- File pesanti (codice, checkpoint, label, input, cache), non persistiti: `/tmp/e03`


In [ ]:
MODE = "infer-46527-s43-zp2"
KIND = "infer"                  # prep | infer
SEG = "pherc0814-46527"                    # nome della label, es. pherc0814-46527
SHORT = "46527"
SEED = 43                    # None nel prep
TAG = "zp2"                    # z13 | zm3 | zp3 (prep) oppure zm5..zp5 (infer)
K = 2                          # offset in slice poolate (None nel prep)
Z_START = 13              # primo piano sorgente della finestra di 84
SOURCE_Z_SLICE = [13, 97]
LAYER_ARGS = "--layer-start 4 --layer-end 21"      # "" per la finestra di default, oppure --layer-start S --layer-end E
EXPECTED_INDICES = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
SETS = "held,train"                # i due segmenti di sviluppo hanno entrambi gli insiemi
WORK = "/kaggle/working/e03"
HEAVY = "/tmp/e03"
SRC_URL = "https://vesuvius-challenge-open-data.s3.amazonaws.com/PHerc0814/segments/20260226000000-46527_2um_try2/surface-volumes/2.399um-0.22m-78keV-volume-20260309142202.zarr"
LABEL_SHAPE = [21, 2130, 3455]
TORCH_EXPECTED = "2.10.0+cu128"
LABEL_TREE_SHA256 = "5659236870d7d0408e330f05f6275bd821fc1d7bdea8c9c8f072dfd4ae8b54f0"
LABEL_FILES, LABEL_BYTES = 1386, 118869
LABEL_ALLOWLIST = {"pherc0139-w016": "a62d3e0ecfc9305758fae3bc0d74d99ecf675bcf846aa910ee4a876ee26ccfd5", "pherc0814-46527": "5659236870d7d0408e330f05f6275bd821fc1d7bdea8c9c8f072dfd4ae8b54f0"}      # nome -> impronta congelata: il sigillo non dipende dai nomi
INPUT_TREE_SHA256 = "bc7423431221bf24b247a8ba80d264b0306f816c52b4ecc0d08115a82305ac52"       # input da usare in questo run (ufficiale o spostato)
OFFICIAL_INPUT_TREE_SHA256 = "bc7423431221bf24b247a8ba80d264b0306f816c52b4ecc0d08115a82305ac52"
SEALED_SEGMENT = "pherc1667-w029"
assert SEG in LABEL_ALLOWLIST and SEG != SEALED_SEGMENT, f"STOP: {SEG} non e' un segmento di sviluppo"
print("MODE", MODE, "SEG", SEG, "SEED", SEED, "TAG", TAG, "K", K, "z_start", Z_START, "layer", LAYER_ARGS or "default")


In [ ]:
# Guardia della lista bianca (revisione R1, finding 4): prima di qualunque lettura di maschera, la cartella
# delle label montata deve essere uno dei due segmenti di sviluppo E con l'impronta congelata. Il sigillo di
# pherc1667-w029 non dipende dal nome di una cartella.
import glob, hashlib, os

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), len(per_file)

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count("/") - 2
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + "/", "(", len(files), "file )")
sealed_hits = glob.glob(f"/kaggle/input/**/{SEALED_SEGMENT}*", recursive=True)
assert not sealed_hits, f"STOP: il segmento sigillato {SEALED_SEGMENT} risulta montato: {sealed_hits[:3]}"
lab_hits = glob.glob(f"/kaggle/input/**/{SEG}/{SEG}_inklabels.zarr/0/.zarray", recursive=True)
assert lab_hits, f"STOP: label {SEG} non montata sotto /kaggle/input"
LABEL_DIR_MOUNTED = os.path.dirname(os.path.dirname(os.path.dirname(lab_hits[0])))
lsha, lfiles = tree_sha256(LABEL_DIR_MOUNTED)
print("label montata:", LABEL_DIR_MOUNTED, "file", lfiles, "tree_sha256", lsha)
assert os.path.basename(LABEL_DIR_MOUNTED) in LABEL_ALLOWLIST, f"STOP: {LABEL_DIR_MOUNTED} fuori dalla lista bianca"
assert lsha == LABEL_ALLOWLIST[SEG] == LABEL_TREE_SHA256, f"STOP: impronta delle label {lsha} diversa da quella congelata"
print("lista bianca superata")


In [ ]:
# Guardia dell'input (run GPU): l'input poolato di questo offset deve essere montato con l'impronta congelata,
# PRIMA di qualunque installazione. Nessun pooling in sessione GPU (piano E02 R1, finding 6).
import glob, json, os
name = f"{SEG}_pooled.zarr" if TAG in ("z0", "zm2", "zp2") else f"{SEG}_pooled_{TAG}.zarr"
hits = glob.glob(f"/kaggle/input/**/{name}/0/.zarray", recursive=True)
assert hits, f"STOP: input {name} non montato sotto /kaggle/input: ripararlo con un run prep (CPU), mai qui"
INPUT_ZARR = os.path.dirname(os.path.dirname(hits[0]))
tsha, nfiles = tree_sha256(INPUT_ZARR)
print("input:", INPUT_ZARR, "file", nfiles, "tree_sha256", tsha)
assert tsha == INPUT_TREE_SHA256, f"STOP: tree_sha256 dell'input montato ({tsha}) diverso da quello congelato ({INPUT_TREE_SHA256})"
import zarr
g = zarr.open(INPUT_ZARR, mode="r")
attrs = dict(g.attrs); print("attrs:", attrs)
assert list(attrs["source_z_slice"]) == SOURCE_Z_SLICE, f"STOP: finestra sorgente {attrs['source_z_slice']} diversa da {SOURCE_Z_SLICE}"
assert tuple(g["0"].shape) == tuple(LABEL_SHAPE), f"STOP: forma {g['0'].shape} diversa dalla label {LABEL_SHAPE}"
open("/kaggle/working/e03_guard.json", "w").write(json.dumps({"input_zarr": INPUT_ZARR, "input_tree_sha256": tsha,
                                                              "input_files": nfiles, "source_z_slice": SOURCE_Z_SLICE}))


In [ ]:
%%bash
# Passo 1 — radice misurabile, cache e temporanei dirottati, guardia dei 15 GB
set -e
mkdir -p /kaggle/working/e03/out /kaggle/working/e03/logs /tmp/e03/tmp /tmp/e03/cache/pip /tmp/e03/cache/hf /tmp/e03/checkpoints /tmp/e03/labels /tmp/e03/input
cat > /kaggle/working/e03/env.sh <<'EOF'
export WORK=/kaggle/working/e03
export HEAVY=/tmp/e03
export TMPDIR=$HEAVY/tmp PIP_CACHE_DIR=$HEAVY/cache/pip HF_HOME=$HEAVY/cache/hf
export LIMIT_BYTES=$((15*1024*1024*1024))
disk_check () {
  local used
  used=$(( $(du -sb "$WORK" | cut -f1) + $(du -sb "$HEAVY" | cut -f1) ))
  echo "spazio_byte=$used ($1)" | tee -a "$WORK/logs/disk_check.log"
  if [ "$used" -gt "$LIMIT_BYTES" ]; then echo "STOP: superati 15 GB ($1)" | tee -a "$WORK/logs/disk_check.log"; exit 1; fi
}
EOF
source /kaggle/working/e03/env.sh
echo "MODE=infer-46527-s43-zp2 SEG=pherc0814-46527 SEED=43 start=$(date -u +%FT%TZ)" > $WORK/logs/run_info.txt
disk_check "inizio"
df -h /kaggle/working /tmp | tail -2


In [ ]:
# Passo 1 (segue) — versioni dell'ambiente e rete verso le sorgenti
import sys, platform, json, urllib.request, torch
env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0}
print(json.dumps(env, indent=1))
json.dump(env, open(f"{WORK}/logs/env_before_install.json", "w"), indent=1)
assert env["torch"] == TORCH_EXPECTED, f"STOP: PyTorch inatteso {env['torch']} (atteso {TORCH_EXPECTED}): Kaggle ha cambiato immagine, aggiornare il piano"
if KIND == "prep":
    assert not env["cuda_available"], "STOP: il run prep deve girare senza acceleratore"
else:
    assert env["cuda_available"], "STOP: run GPU senza CUDA disponibile"
urls = ["https://huggingface.co/api/models/scrollprize/ink_9um", "https://huggingface.co/api/buckets/scrollprize/datasets"]
if KIND == "prep":
    urls.insert(0, SRC_URL + "/2/.zarray")
for url in urls:
    with urllib.request.urlopen(url, timeout=30) as r:
        print(r.status, url[:90]); assert r.status == 200, f"STOP: rete non raggiunge {url}"


In [ ]:
%%bash
# Passo 2 — checkout parziale di villa al commit congelato
set -e
source /kaggle/working/e03/env.sh
cd $HEAVY
[ -d villa/.git ] || git clone -q --filter=blob:none --no-checkout https://github.com/ScrollPrize/villa.git
cd villa
git sparse-checkout init --cone >/dev/null
git sparse-checkout set ink-detection vesuvius >/dev/null
git checkout -q 3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e
HEAD=$(git rev-parse HEAD); echo "villa HEAD=$HEAD" | tee $WORK/logs/villa_commit.txt
[ "$HEAD" = "3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e" ] || { echo "STOP: commit villa diverso"; exit 1; }
ls -l ink-detection/koine_machines/inference/infer.py ink-detection/scripts/prepare_9um_isotropic_input.py vesuvius/pyproject.toml ink-detection/uv.lock
sha256sum ink-detection/scripts/prepare_9um_isotropic_input.py | tee $WORK/logs/prepare_script_sha256.txt
disk_check "dopo checkout"


In [ ]:
# Passo 3 — installazione sul Python di sistema, tutto con --no-deps, PyTorch intatto (cella di E00, invariata)
import subprocess, re, json, sys
VILLA = f"{HEAVY}/villa"
PY = sys.executable

def sh(args):
    return subprocess.run(args, capture_output=True, text=True)

def pip(*args):
    return sh([PY, "-m", "pip", "install", "--no-deps", *args])

def torch_version():
    return sh([PY, "-c", "import torch; print(torch.__version__)"]).stdout.strip()

torch_before = torch_version()
for pkg in ["vesuvius", "ink-detection"]:
    r = pip("-e", f"{VILLA}/{pkg}")
    assert r.returncode == 0, "STOP: pip install fallita\n" + r.stderr[-3000:]
r = pip("zarr==2.18.7", "numcodecs==0.15.1")   # il codice usa l'API Zarr v2
assert r.returncode == 0, "STOP: pip install zarr/numcodecs fallita\n" + r.stderr[-3000:]

lock = open(f"{VILLA}/ink-detection/uv.lock", encoding="utf-8").read()
def locked_version(dist):
    m = re.search(r'\[\[package\]\]\nname = "' + re.escape(dist.lower()) + r'"\nversion = "([^"]+)"', lock)
    return m.group(1) if m else None

ALIAS = {"cv2": "opencv-contrib-python-headless", "PIL": "pillow", "yaml": "pyyaml", "nrrd": "pynrrd",
         "cc3d": "connected-components-3d", "skimage": "scikit-image", "sklearn": "scikit-learn"}
added, numpy_downgraded = [], False

def ensure_import(modname):
    global numpy_downgraded
    for attempt in range(12):
        r = sh([PY, "-c", f"import {modname}"])
        if r.returncode == 0:
            return
        err = r.stderr
        m = re.search(r"No module named '([^'.]+)", err)
        if m:
            mod = m.group(1)
            assert re.fullmatch(r"[A-Za-z0-9_]+", mod), f"STOP: nome di modulo inatteso {mod!r}"
            cands = ([ALIAS[mod]] if mod in ALIAS else []) + [mod, mod.replace("_", "-")]
            dist = next((c for c in cands if locked_version(c)), None)
            assert dist, f"STOP: modulo mancante '{mod}' non presente in uv.lock:\n" + err[-2000:]
            ver = locked_version(dist)
            r2 = pip(f"{dist}=={ver}")
            assert r2.returncode == 0, f"STOP: pip install {dist}=={ver} fallita\n" + r2.stderr[-3000:]
            added.append({"module": mod, "dist": dist, "version": ver}); print("aggiunto", dist, ver, "per", modname)
        elif "numpy" in err.lower() and not numpy_downgraded:
            r2 = pip("numpy<=2.2"); assert r2.returncode == 0, r2.stderr[-3000:]
            numpy_downgraded = True; added.append({"module": "numpy", "dist": "numpy", "version": "<=2.2 (eccezione piano)"})
        else:
            raise AssertionError(f"STOP: import di {modname} fallito per motivo diverso da modulo mancante:\n" + err[-3000:])
    raise AssertionError(f"STOP: import di {modname} ancora fallito dopo i tentativi ammessi")

for modname in ["koine_machines.inference.infer", "vesuvius", "vesuvius.models.build.build_network_from_config",
                "koine_machines.models.make_model"]:
    ensure_import(modname)
for lazy in ["imagecodecs"]:
    if sh([PY, "-c", f"import {lazy}"]).returncode != 0:
        ver = locked_version(lazy); assert ver, f"STOP: {lazy} non presente in uv.lock"
        r2 = pip(f"{lazy}=={ver}"); assert r2.returncode == 0, f"STOP: pip install {lazy}=={ver} fallita\n" + r2.stderr[-3000:]
        added.append({"module": lazy, "dist": lazy, "version": ver}); print("aggiunto", lazy, ver, "(dipendenza pigra di tifffile)")
assert len(added) <= 10, f"STOP: {len(added)} pacchetti aggiunti, oltre il limite di dieci del piano"

torch_after = torch_version()
assert torch_after == torch_before == TORCH_EXPECTED, f"STOP: PyTorch cambiato da {torch_before} a {torch_after}"
helptxt = sh([PY, "-m", "koine_machines.inference.infer", "--help"]).stdout
for flag in ["--no-compile", "--gpus", "--layer-start"]:
    assert flag in helptxt, f"STOP: opzione {flag} assente nell'entry point"
hf = sh(["hf", "--version"])
if hf.returncode != 0:
    r2 = pip("huggingface_hub"); assert r2.returncode == 0, r2.stderr[-2000:]
    hf = sh(["hf", "--version"]); added.append({"module": "hf", "dist": "huggingface_hub", "version": hf.stdout.strip()})
pipl = [l for l in sh([PY, "-m", "pip", "list"]).stdout.splitlines()
        if re.match(r"(?i)^(torch|torchvision|zarr|numcodecs|numpy|tifffile|timm|scipy|fsspec|s3fs|aiohttp|huggingface.hub|monai|koine.machines|vesuvius|albumentations|einops|opencv|imagecodecs) ", l)]
open(f"{WORK}/logs/pip_versions.txt", "w").write("\n".join(pipl) + "\n")
info = {"torch_before": torch_before, "torch_after": torch_after, "added_packages": added, "hf_version": hf.stdout.strip()}
json.dump(info, open(f"{WORK}/logs/install.json", "w"), indent=1)
print(json.dumps(info, indent=1)); print("\n".join(pipl))


In [ ]:
%%bash
# Passo 4 — checkpoint con verifica esatta di hash e dimensioni (cella di E00)
set -e
source /kaggle/working/e03/env.sh
cd $HEAVY
disk_check "prima dei checkpoint"
hf download scrollprize/ink_9um hybrid_3d2d-seed42/step-075000.pth hybrid_3d2d-seed43/step-075000.pth \
   --revision 7109667e2607db1b90c37c8b09cb876ea7fe7bb1 --local-dir checkpoints/ink_9um > /dev/null
sha256sum checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth checkpoints/ink_9um/hybrid_3d2d-seed43/step-075000.pth | tee $WORK/logs/checkpoints_sha256.txt
stat -c "%s %n" checkpoints/ink_9um/hybrid_3d2d-seed4*/step-075000.pth | tee -a $WORK/logs/checkpoints_sha256.txt
grep -q "^e635558ae6a1a807a7e5ec1e83adfd45bc3c0ac53883ea43f1d4e085d62a9cab  checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth" $WORK/logs/checkpoints_sha256.txt || { echo "STOP: SHA-256 seed42 diverso"; exit 1; }
grep -q "^2aeaa85a35ef28d7bc7bf3e848c4a6a91385e9132710927fdba41133c4ecb28f  checkpoints/ink_9um/hybrid_3d2d-seed43/step-075000.pth" $WORK/logs/checkpoints_sha256.txt || { echo "STOP: SHA-256 seed43 diverso"; exit 1; }
disk_check "dopo i checkpoint"


In [ ]:
# Passo 4 (segue) — label del segmento dal dataset Kaggle privato (ricerca ricorsiva: il mount cambia fra sessioni
# CPU e GPU, lezione di E00), verificata file per file contro manifest.json e per contenuto (tree_sha256);
# ripiego: download diretto dal bucket con 4 thread e attesa crescente (HTTP 429), come E00.
import hashlib, os, shutil, json, glob, re, time, urllib.request, concurrent.futures as cf
PREFIX = f"ink_9um/labels/aligned-scrollprizeorg-21slices/{SEG}/"
API = "https://huggingface.co/api/buckets/scrollprize/datasets/tree/" + PREFIX.rstrip("/")
RESOLVE = "https://huggingface.co/buckets/scrollprize/datasets/resolve/"
DEST = f"{HEAVY}/labels/{SEG}"

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), per_file

def check_label_tree(dest, source_desc):
    n = sum(len(fs) for _, _, fs in os.walk(dest)); b = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(dest) for f in fs)
    assert (n, b) == (LABEL_FILES, LABEL_BYTES), f"STOP: label con {n} file / {b} byte, attesi {(LABEL_FILES, LABEL_BYTES)}"
    tsha, _ = tree_sha256(dest)
    assert tsha == LABEL_TREE_SHA256, f"STOP: contenuto della label diverso da quello congelato (tree sha256 {tsha})"
    open(f"{WORK}/logs/label_count.txt", "w").write(f"seg={SEG} file={n} byte={b} tree_sha256={tsha} source={source_desc}\n")
    za = json.load(open(f"{dest}/{SEG}_inklabels.zarr/0/.zarray")); open(f"{WORK}/logs/label_zarray.json", "w").write(json.dumps(za))
    assert za["shape"] == LABEL_SHAPE, f"STOP: forma della label {za['shape']} diversa da {LABEL_SHAPE}"
    print(f"label verificata: file={n} byte={b} tree_sha256={tsha} ({source_desc}) shape={za['shape']}")

roots = glob.glob(f"/kaggle/input/**/{SEG}/{SEG}_inklabels.zarr/0/.zarray", recursive=True)
manifests = [m for m in glob.glob("/kaggle/input/**/manifest.json", recursive=True) if SEG in json.load(open(m)).get("segments", {})]
print("label montata:", roots[:1], "| manifest:", manifests[:1])
if os.path.isdir(DEST):
    shutil.rmtree(DEST)
if roots and manifests:
    src = os.path.dirname(os.path.dirname(os.path.dirname(roots[0])))       # .../<SEG>
    man = json.load(open(manifests[0]))["segments"][SEG]
    files = man["files"]
    assert (len(files), sum(f["size"] for f in files)) == (LABEL_FILES, LABEL_BYTES), "STOP: manifest del dataset diverso dalle costanti congelate"
    missing = [f["path"] for f in files if not (os.path.isfile(os.path.join(src, f["path"][len(PREFIX):])) and os.path.getsize(os.path.join(src, f["path"][len(PREFIX):])) == f["size"])]
    assert not missing, f"STOP: {len(missing)} file della label mancanti o di dimensione diversa, p.es. {missing[:3]}"
    shutil.copytree(src, DEST)
    check_label_tree(DEST, f"kaggle_dataset manifest_sha256={hashlib.sha256(open(manifests[0], 'rb').read()).hexdigest()}")
else:
    assert KIND == "prep", "STOP: nei run GPU la label deve essere montata (nessun ripiego di rete con la GPU allocata)"
    def list_label_files():
        files, url = [], API
        while url:
            req = urllib.request.Request(url, headers={"User-Agent": "papyruslab-e02"})
            with urllib.request.urlopen(req, timeout=60) as r:
                files += [(e["path"], int(e["size"])) for e in json.load(r) if e.get("type") == "file"]
                m = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", "") or "")
                url = m.group(1) if m else None
        return files
    def fetch(item):
        path, size = item
        assert path.startswith(PREFIX) and ".." not in path, f"STOP: percorso inatteso {path}"
        out = os.path.join(DEST, path[len(PREFIX):])
        if os.path.exists(out) and os.path.getsize(out) == size:
            return size
        os.makedirs(os.path.dirname(out), exist_ok=True)
        last = None
        for attempt in range(8):
            try:
                req = urllib.request.Request(RESOLVE + path, headers={"User-Agent": "papyruslab-e02"})
                with urllib.request.urlopen(req, timeout=60) as r:
                    data = r.read()
                if len(data) == size:
                    open(out, "wb").write(data); return size
                last = f"dimensione {len(data)} != {size}"
            except Exception as ex:
                last = ex
            time.sleep(min(60, 5 * 2 ** attempt))
        raise RuntimeError(f"STOP: download fallito per {path}: {last}")
    files = list_label_files(); total = sum(s for _, s in files)
    assert (len(files), total) == (LABEL_FILES, LABEL_BYTES), f"STOP: label diversa dalla misura congelata: {len(files)} file, {total} byte"
    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=4) as ex:
        got = sum(ex.map(fetch, files))
    print(f"scaricati {got} byte in {time.time() - t0:.0f} s")
    check_label_tree(DEST, "direct_download")


In [ ]:
# Passo 5 (infer) — costruire il modello dal checkpoint su CPU come fa infer.py, in sottoprocesso (cella di E00)
import subprocess, sys, json
code = r'''
import argparse, json, torch
from koine_machines.inference import infer as kinfer
args = argparse.Namespace(checkpoint="__CKPT__", amp_dtype="auto", model_type="auto", metadata_json=None)
cm = kinfer.configure_model(args)
info = {"in_chans": int(cm.in_chans), "amp_dtype": str(cm.amp_dtype),
        "n_params": int(sum(p.numel() for p in cm.model.parameters())),
        "model_class": type(cm.model).__name__, "preprocessing": str(cm.preprocessing)[:200]}
print("MODEL_BUILD_JSON=" + json.dumps(info))
'''.replace("__CKPT__", f"{HEAVY}/checkpoints/ink_9um/hybrid_3d2d-seed{SEED}/step-075000.pth")
r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:])
assert r.returncode == 0, "STOP: costruzione del modello su CPU fallita (vedi stderr sopra)"
info = json.loads(r.stdout.split("MODEL_BUILD_JSON=")[1].splitlines()[0])
json.dump(info, open(f"{WORK}/logs/model_build_cpu.json", "w"), indent=1)
assert info["in_chans"] == 17, "STOP: il checkpoint non dichiara 17 slice in ingresso"
assert "float16" in info["amp_dtype"], f"STOP: AMP dtype inatteso {info['amp_dtype']}"
print("modello costruito su CPU:", info)


In [ ]:
%%bash
# Passo 6/9 — inferenza seed 43 con la finestra Z dell'offset zp2, una sola T4
source /kaggle/working/e03/env.sh
cd $HEAVY
INPUT_ZARR=$(python -c "import json; print(json.load(open('/kaggle/working/e03_guard.json'))['input_zarr'])")
echo "input=$INPUT_ZARR" | tee $WORK/logs/input_path.txt
disk_check "prima inferenza seed43 zp2"
nvidia-smi --query-gpu=index,name,memory.total,memory.used,driver_version --format=csv > $WORK/logs/nvidia-smi-before.txt \
  || { echo "STOP: nvidia-smi non disponibile: nessuna GPU assegnata"; exit 1; }
cat $WORK/logs/nvidia-smi-before.txt
export CUDA_VISIBLE_DEVICES=0
python -c "import torch; assert torch.cuda.is_available(); print(torch.__version__, 'visibili', torch.cuda.device_count(), torch.cuda.get_device_name(0))" | tee -a $WORK/logs/env_gpu.txt \
  || { echo "STOP: PyTorch non vede la GPU"; exit 1; }
nvidia-smi --query-gpu=timestamp,index,memory.used --format=csv,noheader,nounits -l 5 > $WORK/logs/gpu_samples.csv &
SAMPLER=$!
set -o pipefail
START=$(date +%s)
timeout -s INT -k 30 1800 python -m koine_machines.inference.infer \
  "$INPUT_ZARR" checkpoints/ink_9um/hybrid_3d2d-seed43/step-075000.pth $WORK/out/pherc0814-46527_seed43_step075000_zp2.tif \
  --overlap 0.5 --blend-mode hann --no-compile --gpus 0 --batch-size 1 --layer-start 4 --layer-end 21 \
  2>&1 | tee $WORK/logs/infer_seed43zp2.log
EXIT=${PIPESTATUS[0]}; END=$(date +%s)
kill $SAMPLER 2>/dev/null; wait $SAMPLER 2>/dev/null
CAUSA=normale; [ "$EXIT" -eq 124 ] && CAUSA=timeout_1800s
echo "exit_code=$EXIT durata_s=$((END-START)) causa=$CAUSA" | tee -a $WORK/logs/infer_seed43zp2.log
nvidia-smi --query-gpu=index,memory.used --format=csv > $WORK/logs/nvidia-smi-after.txt; cat $WORK/logs/nvidia-smi-after.txt
disk_check "dopo inferenza"


In [ ]:
# Passo 6/9 (segue) — controlli sul log: indici attesi per QUESTO offset, canali, GPU singola misurata
import csv, re
log = open(f"{WORK}/logs/infer_seed{SEED}_{TAG}.log", encoding="utf-8", errors="replace").read()
m = re.search(r"exit_code=(\d+) durata_s=(\d+) causa=(\S+)", log); assert m, "STOP: riga finale di esito assente nel log"
exit_code, durata, causa = int(m.group(1)), int(m.group(2)), m.group(3)
print("exit_code", exit_code, "durata_s", durata, "causa", causa)
assert exit_code == 0, f"STOP: inferenza terminata con exit_code={exit_code} ({causa})"
expected = "Selected source layer indices=" + str(EXPECTED_INDICES)
assert expected in log, f"STOP: indici di layer diversi da quelli attesi per {TAG}: {EXPECTED_INDICES}"
assert "in_chans=17" in log, "STOP: in_chans diverso da 17"
assert "Using CUDA device 0 for inference." in log, "STOP: il log non conferma l'uso del solo device 0"
assert re.search(r"Wrote .*%s_seed%d_step075000_%s.tif" % (SEG, SEED, TAG), log), "STOP: il log non conferma la scrittura del TIFF"
before = open(f"{WORK}/logs/nvidia-smi-before.txt").read()
gpus_before = [l for l in before.splitlines()[1:] if l.strip()]
assert gpus_before and all("T4" in l for l in gpus_before), f"STOP: GPU inattese o assenti: {gpus_before}"
peak = {}
for row in csv.reader(open(f"{WORK}/logs/gpu_samples.csv")):
    if len(row) >= 3 and row[1].strip().isdigit():
        idx, mem = int(row[1]), int(row[2]); peak[idx] = max(peak.get(idx, 0), mem)
print("picco memoria per GPU (MiB):", peak, "| GPU allocate:", len(gpus_before))
assert peak.get(0, 0) > 200, "STOP: nessun uso misurato della GPU 0"
for idx in peak:
    if idx != 0:
        assert peak[idx] < 100, f"STOP: la GPU {idx} ha usato {peak[idx]} MiB: vincolo di una sola GPU non rispettato"
open(f"{WORK}/logs/gpu_peak.json", "w").write(str(peak))
GPU_DURATION_S = durata


In [ ]:
# Passo 6/9 (segue) — metriche del punto della curva con gli script congelati, inlineati dal generatore.
# e02_metrics resta congelato; e03_metrics lo avvolge e aggiunge il blocco di identita' e03_point.
import base64, importlib, json, hashlib, os, sys
from pathlib import Path
repo = f"{WORK}/repo"
os.makedirs(f"{repo}/scripts", exist_ok=True); os.makedirs(f"{repo}/configs/e03", exist_ok=True)
for name, b64 in (("e02_metrics.py", "..."), ("tree_sha256.py", "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQcmludCB0aGUgY29udGVudCBoYXNoIG9mIGEgZGlyZWN0b3J5IHRyZWUgKEUwMiBoZWxwZXIsIHNhbWUgZGVmaW5pdGlvbiBldmVyeXdoZXJlIGluIFBhcHlydXNMYWIpLgoKRGVmaW5pdGlvbiAoc2NyaXB0cy9idWlsZF93MDM1X2xhYmVsX2RhdGFzZXQucHksIEUwMCk6IHNoYTI1NiBvdmVyIHRoZSBzb3J0ZWQgbGluZXMgJ3JlbHBhdGhcXG5zaGEyNTYoZmlsZSlcXG4nLApyZWxwYXRoIHJlbGF0aXZlIHRvIHRoZSBnaXZlbiBkaXJlY3RvcnksIFBPU0lYIHNlcGFyYXRvcnMuIFR3byB0cmVlcyB3aXRoIHRoZSBzYW1lIGhhc2ggaGF2ZSBpZGVudGljYWwgZmlsZXMuCgpVc2FnZToKICBweXRob24gc2NyaXB0cy90cmVlX3NoYTI1Ni5weSA8ZGlyZWN0b3J5PiBbLS1saXN0XSAgICAgIyAtLWxpc3QgYWxzbyBwcmludHMgb25lIGxpbmUgcGVyIGZpbGUKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgdHJlZV9zaGEyNTYocm9vdDogUGF0aCkgLT4gdHVwbGVbc3RyLCBkaWN0W3N0ciwgc3RyXV06CiAgICBwZXJfZmlsZSA9IHt9CiAgICBmb3IgcCBpbiBzb3J0ZWQoeCBmb3IgeCBpbiByb290LnJnbG9iKCIqIikgaWYgeC5pc19maWxlKCkpOgogICAgICAgIHBlcl9maWxlW3AucmVsYXRpdmVfdG8ocm9vdCkuYXNfcG9zaXgoKV0gPSBoYXNobGliLnNoYTI1NihwLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICBmb3IgcmVsLCBkaWdlc3QgaW4gc29ydGVkKHBlcl9maWxlLml0ZW1zKCkpOgogICAgICAgIGgudXBkYXRlKGYie3JlbH1cbntkaWdlc3R9XG4iLmVuY29kZSgpKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCksIHBlcl9maWxlCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLCBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgYXAuYWRkX2FyZ3VtZW50KCJkaXJlY3RvcnkiLCB0eXBlPVBhdGgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGlzdCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICByb290ID0gYS5kaXJlY3RvcnkucmVzb2x2ZSgpCiAgICBhc3NlcnQgcm9vdC5pc19kaXIoKSwgZiJub3QgYSBkaXJlY3Rvcnk6IHtyb290fSIKICAgIGRpZ2VzdCwgcGVyX2ZpbGUgPSB0cmVlX3NoYTI1Nihyb290KQogICAgaWYgYS5saXN0OgogICAgICAgIGZvciByZWwsIGQgaW4gc29ydGVkKHBlcl9maWxlLml0ZW1zKCkpOgogICAgICAgICAgICBwcmludChkLCByZWwpCiAgICB0b3RhbCA9IHN1bSgocm9vdCAvIHJlbCkuc3RhdCgpLnN0X3NpemUgZm9yIHJlbCBpbiBwZXJfZmlsZSkKICAgIHByaW50KGYidHJlZV9zaGEyNTY9e2RpZ2VzdH0gZmlsZXM9e2xlbihwZXJfZmlsZSl9IGJ5dGVzPXt0b3RhbH0gcm9vdD17cm9vdH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"),
                  ("e03_metrics.py", "...")):
    src = base64.b64decode(b64).decode("utf-8")
    open(f"{repo}/scripts/{name}", "w", encoding="utf-8", newline="\n").write(src)
    open(f"{WORK}/logs/{name}.sha256", "w").write(hashlib.sha256(src.encode("utf-8")).hexdigest() + "\n")
open(f"{repo}/configs/e03/offsets.json", "w", encoding="utf-8", newline="\n").write(
    base64.b64decode("ewogIl9ub3RlIjogIk1hdHJpY2UgZGVnbGkgb2Zmc2V0IFogZGkgRTAzIChwaWFubyBkb2NzL3BsYW5zLzIwMjYtMDktMDctZTAzLXRvbGxlcmFuemEtb2Zmc2V0LXoubWQsIHNlemlvbmUgMi4yIGUgcGFzc28gMSkuIEZhIGZlZGUgcXVlc3RvIGZpbGUgaW5zaWVtZSBhbCBwaWFubyBjb25nZWxhdG86IGlsIGdlbmVyYXRvcmUgZGVpIG5vdGVib29rIGhhIGF1dG9yaXRhJyBzb2xvIHN1bGwnZXNwYW5zaW9uZSBtZWNjYW5pY2EgKHJldmlzaW9uZSBSMSwgZmluZGluZyA3KS4gayA9IG9mZnNldCBpbiBzbGljZSBwb29sYXRlOyB1bmEgc2xpY2UgcG9vbGF0YSA9IDQgcGlhbmkgc29yZ2VudGUgZGEgMiwzOTkgdW0gPSA5LDU5NiB1bS4iLAogInBsYW4iOiAiZG9jcy9wbGFucy8yMDI2LTA5LTA3LWUwMy10b2xsZXJhbnphLW9mZnNldC16Lm1kIiwKICJzbGljZV91bSI6IDkuNTk2LAogInNvdXJjZV9wbGFuZXMiOiAxMDksCiAicG9vbGVkX3NsaWNlcyI6IDIxLAogIm1vZGVsX3NsaWNlcyI6IDE3LAogIm9mZmljaWFsX3pfc3RhcnQiOiAxMywKICJhbm5vdGF0aW9uX2NlbnRlcl9jaGFubmVsIjogMTAsCiAiY2VudHJlX3BsYW5lX2F0X2swIjogNTUsCiAiel9zdGFydF9hbGxvd2VkIjogWwogIDAsCiAgMjUKIF0sCiAic2VnbWVudHMiOiBbCiAgInBoZXJjMDEzOS13MDE2IiwKICAicGhlcmMwODE0LTQ2NTI3IgogXSwKICJzZWVkcyI6IFsKICA0MiwKICA0MwogXSwKICJzZWFsZWRfc2VnbWVudF9uZXZlcl90b3VjaGVkIjogInBoZXJjMTY2Ny13MDI5IiwKICJvZmZzZXRzIjogWwogIHsKICAgImsiOiAtNSwKICAgInRhZyI6ICJ6bTUiLAogICAic3RhZ2UiOiAyLAogICAiaW5wdXQiOiAic2hpZnRlZF9tMyIsCiAgICJwb29sX3NoaWZ0X3NsaWNlcyI6IC0zLAogICAiel9zdGFydCI6IDEsCiAgICJzb3VyY2Vfel9zbGljZSI6IFsKICAgIDEsCiAgICA4NQogICBdLAogICAibGF5ZXJfc3RhcnQiOiAwLAogICAibGF5ZXJfZW5kIjogMTcsCiAgICJleHBlY3RlZF9pbmRpY2VzIjogWwogICAgMCwKICAgIDEsCiAgICAyLAogICAgMywKICAgIDQsCiAgICA1LAogICAgNiwKICAgIDcsCiAgICA4LAogICAgOSwKICAgIDEwLAogICAgMTEsCiAgICAxMiwKICAgIDEzLAogICAgMTQsCiAgICAxNSwKICAgIDE2CiAgIF0sCiAgICJzb3VyY2VfcGxhbmVzX3NlZW4iOiBbCiAgICAxLAogICAgNjkKICAgXSwKICAgImNlbnRyZV9wbGFuZSI6IDM1LAogICAibWljcm9tZXRyZXMiOiAtNDcuOTgsCiAgICJvcmllbnRhdGlvbl9nYXRlIjogInJlZ2lzdHJhdG8iCiAgfSwKICB7CiAgICJrIjogLTMsCiAgICJ0YWciOiAiem0zIiwKICAgInN0YWdlIjogMiwKICAgImlucHV0IjogInNoaWZ0ZWRfbTMiLAogICAicG9vbF9zaGlmdF9zbGljZXMiOiAtMywKICAgInpfc3RhcnQiOiAxLAogICAic291cmNlX3pfc2xpY2UiOiBbCiAgICAxLAogICAgODUKICAgXSwKICAgImxheWVyX3N0YXJ0IjogbnVsbCwKICAgImxheWVyX2VuZCI6IG51bGwsCiAgICJleHBlY3RlZF9pbmRpY2VzIjogWwogICAgMiwKICAgIDMsCiAgICA0LAogICAgNSwKICAgIDYsCiAgICA3LAogICAgOCwKICAgIDksCiAgICAxMCwKICAgIDExLAogICAgMTIsCiAgICAxMywKICAgIDE0LAogICAgMTUsCiAgICAxNiwKICAgIDE3LAogICAgMTgKICAgXSwKICAgInNvdXJjZV9wbGFuZXNfc2VlbiI6IFsKICAgIDksCiAgICA3NwogICBdLAogICAiY2VudHJlX3BsYW5lIjogNDMsCiAgICJtaWNyb21ldHJlcyI6IC0yOC43ODgsCiAgICJvcmllbnRhdGlvbl9nYXRlIjogInJlZ2lzdHJhdG8iCiAgfSwKICB7CiAgICJrIjogLTIsCiAgICJ0YWciOiAiem0yIiwKICAgInN0YWdlIjogMSwKICAgImlucHV0IjogIm9mZmljaWFsIiwKICAgInBvb2xfc2hpZnRfc2xpY2VzIjogMCwKICAgInpfc3RhcnQiOiAxMywKICAgInNvdXJjZV96X3NsaWNlIjogWwogICAgMTMsCiAgICA5NwogICBdLAogICAibGF5ZXJfc3RhcnQiOiAwLAogICAibGF5ZXJfZW5kIjogMTcsCiAgICJleHBlY3RlZF9pbmRpY2VzIjogWwogICAgMCwKICAgIDEsCiAgICAyLAogICAgMywKICAgIDQsCiAgICA1LAogICAgNiwKICAgIDcsCiAgICA4LAogICAgOSwKICAgIDEwLAogICAgMTEsCiAgICAxMiwKICAgIDEzLAogICAgMTQsCiAgICAxNSwKICAgIDE2CiAgIF0sCiAgICJzb3VyY2VfcGxhbmVzX3NlZW4iOiBbCiAgICAxMywKICAgIDgxCiAgIF0sCiAgICJjZW50cmVfcGxhbmUiOiA0NywKICAgIm1pY3JvbWV0cmVzIjogLTE5LjE5MiwKICAgIm9yaWVudGF0aW9uX2dhdGUiOiAiYmxvY2NhbnRlIgogIH0sCiAgewogICAiayI6IDAsCiAgICJ0YWciOiAiejAiLAogICAic3RhZ2UiOiAwLAogICAiaW5wdXQiOiAib2ZmaWNpYWwiLAogICAicG9vbF9zaGlmdF9zbGljZXMiOiAwLAogICAiel9zdGFydCI6IDEzLAogICAic291cmNlX3pfc2xpY2UiOiBbCiAgICAxMywKICAgIDk3CiAgIF0sCiAgICJsYXllcl9zdGFydCI6IG51bGwsCiAgICJsYXllcl9lbmQiOiBudWxsLAogICAiZXhwZWN0ZWRfaW5kaWNlcyI6IFsKICAgIDIsCiAgICAzLAogICAgNCwKICAgIDUsCiAgICA2LAogICAgNywKICAgIDgsCiAgICA5LAogICAgMTAsCiAgICAxMSwKICAgIDEyLAogICAgMTMsCiAgICAxNCwKICAgIDE1LAogICAgMTYsCiAgICAxNywKICAgIDE4CiAgIF0sCiAgICJzb3VyY2VfcGxhbmVzX3NlZW4iOiBbCiAgICAyMSwKICAgIDg5CiAgIF0sCiAgICJjZW50cmVfcGxhbmUiOiA1NSwKICAgIm1pY3JvbWV0cmVzIjogMC4wLAogICAib3JpZW50YXRpb25fZ2F0ZSI6ICJiYXNlbGluZSBFMDIiCiAgfSwKICB7CiAgICJrIjogMiwKICAgInRhZyI6ICJ6cDIiLAogICAic3RhZ2UiOiAxLAogICAiaW5wdXQiOiAib2ZmaWNpYWwiLAogICAicG9vbF9zaGlmdF9zbGljZXMiOiAwLAogICAiel9zdGFydCI6IDEzLAogICAic291cmNlX3pfc2xpY2UiOiBbCiAgICAxMywKICAgIDk3CiAgIF0sCiAgICJsYXllcl9zdGFydCI6IDQsCiAgICJsYXllcl9lbmQiOiAyMSwKICAgImV4cGVjdGVkX2luZGljZXMiOiBbCiAgICA0LAogICAgNSwKICAgIDYsCiAgICA3LAogICAgOCwKICAgIDksCiAgICAxMCwKICAgIDExLAogICAgMTIsCiAgICAxMywKICAgIDE0LAogICAgMTUsCiAgICAxNiwKICAgIDE3LAogICAgMTgsCiAgICAxOSwKICAgIDIwCiAgIF0sCiAgICJzb3VyY2VfcGxhbmVzX3NlZW4iOiBbCiAgICAyOSwKICAgIDk3CiAgIF0sCiAgICJjZW50cmVfcGxhbmUiOiA2MywKICAgIm1pY3JvbWV0cmVzIjogMTkuMTkyLAogICAib3JpZW50YXRpb25fZ2F0ZSI6ICJibG9jY2FudGUiCiAgfSwKICB7CiAgICJrIjogMywKICAgInRhZyI6ICJ6cDMiLAogICAic3RhZ2UiOiAyLAogICAiaW5wdXQiOiAic2hpZnRlZF9wMyIsCiAgICJwb29sX3NoaWZ0X3NsaWNlcyI6IDMsCiAgICJ6X3N0YXJ0IjogMjUsCiAgICJzb3VyY2Vfel9zbGljZSI6IFsKICAgIDI1LAogICAgMTA5CiAgIF0sCiAgICJsYXllcl9zdGFydCI6IG51bGwsCiAgICJsYXllcl9lbmQiOiBudWxsLAogICAiZXhwZWN0ZWRfaW5kaWNlcyI6IFsKICAgIDIsCiAgICAzLAogICAgNCwKICAgIDUsCiAgICA2LAogICAgNywKICAgIDgsCiAgICA5LAogICAgMTAsCiAgICAxMSwKICAgIDEyLAogICAgMTMsCiAgICAxNCwKICAgIDE1LAogICAgMTYsCiAgICAxNywKICAgIDE4CiAgIF0sCiAgICJzb3VyY2VfcGxhbmVzX3NlZW4iOiBbCiAgICAzMywKICAgIDEwMQogICBdLAogICAiY2VudHJlX3BsYW5lIjogNjcsCiAgICJtaWNyb21ldHJlcyI6IDI4Ljc4OCwKICAgIm9yaWVudGF0aW9uX2dhdGUiOiAicmVnaXN0cmF0byIKICB9LAogIHsKICAgImsiOiA1LAogICAidGFnIjogInpwNSIsCiAgICJzdGFnZSI6IDIsCiAgICJpbnB1dCI6ICJzaGlmdGVkX3AzIiwKICAgInBvb2xfc2hpZnRfc2xpY2VzIjogMywKICAgInpfc3RhcnQiOiAyNSwKICAgInNvdXJjZV96X3NsaWNlIjogWwogICAgMjUsCiAgICAxMDkKICAgXSwKICAgImxheWVyX3N0YXJ0IjogNCwKICAgImxheWVyX2VuZCI6IDIxLAogICAiZXhwZWN0ZWRfaW5kaWNlcyI6IFsKICAgIDQsCiAgICA1LAogICAgNiwKICAgIDcsCiAgICA4LAogICAgOSwKICAgIDEwLAogICAgMTEsCiAgICAxMiwKICAgIDEzLAogICAgMTQsCiAgICAxNSwKICAgIDE2LAogICAgMTcsCiAgICAxOCwKICAgIDE5LAogICAgMjAKICAgXSwKICAgInNvdXJjZV9wbGFuZXNfc2VlbiI6IFsKICAgIDQxLAogICAgMTA5CiAgIF0sCiAgICJjZW50cmVfcGxhbmUiOiA3NSwKICAgIm1pY3JvbWV0cmVzIjogNDcuOTgsCiAgICJvcmllbnRhdGlvbl9nYXRlIjogInJlZ2lzdHJhdG8iCiAgfQogXQp9Cg==").decode("utf-8"))
sys.path.insert(0, f"{repo}/scripts")
e03m = importlib.import_module("e03_metrics")
tif = Path(f"{WORK}/out/{SEG}_seed{SEED}_step075000_{TAG}.tif")
rep = e03m.build_run_report(tif, Path(LABEL_DIR_MOUNTED), k=K, seed=SEED, threshold=91,
                            input_tree_sha256=INPUT_TREE_SHA256, layer_indices=EXPECTED_INDICES,
                            source_z_slice=SOURCE_Z_SLICE, run_id="E03-R01", sets=("held", "train"))
assert rep["shape_ok"], f"STOP: TIFF {rep['shape']} {rep['dtype']} diverso dalla label"
train = rep["sets"]["train"]
ok_orient = train["orientation"]["orientamento_ok"]
disjoint = rep["disjoint_check"]["n_px_held_and_train"]
gate_A = "superato"
# L'orientamento e' bloccante solo sugli input ufficiali (|k| = 2): per gli input spostati l'allineamento XY e'
# gia' provato dall'uguaglianza slice a slice nel run prep, e a offset grandi il segnale puo' degradarsi.
blocking = (TAG in ("zm2", "zp2"))
if blocking:
    gate_B = "superato" if (ok_orient is True and disjoint == 0) else ("non_valutabile" if ok_orient is None else "fallito")
else:
    gate_B = "superato" if disjoint == 0 else "fallito"
rep.update({"gate_A": gate_A, "gate_B": gate_B, "gate_B_orientation_blocking": blocking,
            "gpu_duration_s": GPU_DURATION_S, "mode": MODE})
json.dump(rep, open(f"{WORK}/out/metrics_{SEG}_s{SEED}_{TAG}.json", "w"), indent=1, sort_keys=True, default=float)
h = rep["sets"]["held"]
print("AUROC held", h["auroc"], "| bestF1", h["best_f1"], "| F1@91", h["at_threshold"]["f1"])
print("AUROC train", train["auroc"], "| orientamento", ok_orient, "(bloccante:", blocking, ") | held&train", disjoint)
print("GATE A:", gate_A, "| GATE B:", gate_B)


In [ ]:
%%bash
# Persistenza — hash di tutto cio' che viene conservato, stato finale
set -e
source /kaggle/working/e03/env.sh
cp /kaggle/working/e03/env.sh $WORK/logs/env.sh.txt
[ -f /kaggle/working/e03_guard.json ] && cp /kaggle/working/e03_guard.json $WORK/logs/guard.json
echo "end=$(date -u +%FT%TZ)" >> $WORK/logs/run_info.txt
disk_check "finale"
cd $WORK && find out logs -type f ! -name SHA256SUMS -print0 | sort -z | xargs -0 sha256sum > out/SHA256SUMS
cat out/SHA256SUMS
echo "persistito: $(du -sh $WORK | cut -f1)"


In [ ]:
# Verdetto del run: dopo la persistenza, un gate non superato rende il run 'error'
import json
res = json.load(open(f"{WORK}/out/metrics_{SEG}_s{SEED}_{TAG}.json"))
print("gate_A =", res["gate_A"], "| gate_B =", res["gate_B"], "| AUROC held =", res["sets"]["held"]["auroc"])
assert res["gate_A"] == "superato" and res["gate_B"] == "superato", \
    f"E03 run {MODE} NON superato: gate_A={res['gate_A']} gate_B={res['gate_B']} (metriche e log persistiti)"
